# Module 5 — Your First Models
**FissionLab · AI/ML Foundations · Aarush**

Weeks 8, 9, and 10:
- **W8 (Jul 20):** HOML Ch. 2 — end-to-end project on a real dataset
- **W9 (Jul 27):** Linear regression from scratch + scikit-learn (HOML Ch. 4)
- **W10 (Aug 3):** Classification — logistic regression, confusion matrix, precision/recall (HOML Ch. 3)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing, load_breast_cancer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error, r2_score, accuracy_score,
    confusion_matrix, classification_report
)

## Week 8 — End-to-End ML Project (HOML Ch. 2)
Follow the full pipeline: load → explore → prepare → train → evaluate.
We'll use the California Housing dataset (similar to HOML's housing walkthrough).

In [ ]:
import pandas as pd

housing = fetch_california_housing(as_frame=True)
df = housing.frame
print('Shape:', df.shape)
print(df.describe().round(2))
print('\nNull counts:\n', df.isnull().sum())

In [ ]:
# Feature engineering (HOML Ch. 2 pattern)
df['rooms_per_household'] = df['AveRooms'] / df['HouseAge'].clip(lower=1)
df['bedrooms_per_room'] = df['AveBedrms'] / df['AveRooms'].clip(lower=1)
df['pop_per_household'] = df['Population'] / df['AveOccup'].clip(lower=1)

feature_cols = ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms',
                 'Population', 'AveOccup', 'Latitude', 'Longitude',
                 'rooms_per_household', 'bedrooms_per_room', 'pop_per_household']
X = df[feature_cols].values
y = df['MedHouseVal'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, y_pred)

print(f'RMSE: {rmse:.4f}  (predictions off by ~${rmse*100000:.0f} on average)')
print(f'R²:   {r2:.4f}  (model explains {r2*100:.1f}% of variance in prices)')

plt.figure(figsize=(7, 5))
plt.scatter(y_test[:500], y_pred[:500], alpha=0.3, s=15, color='steelblue')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
          'r--', lw=2, label='Perfect prediction')
plt.title('Predicted vs Actual House Value')
plt.xlabel('Actual'); plt.ylabel('Predicted')
plt.legend(); plt.show()

---
## Week 9 — Linear Regression from Scratch vs sklearn (HOML Ch. 4)

In [ ]:
np.random.seed(42)
X_s = 2 * np.random.rand(100, 1)
y_s = 4 + 3 * X_s.ravel() + np.random.randn(100)  # true: intercept=4, slope=3
X_b = np.c_[np.ones((100, 1)), X_s]

# Method 1: gradient descent (from scratch)
w = np.random.randn(2)
lr = 0.1
for _ in range(1000):
    grad = (2 / len(X_b)) * X_b.T @ (X_b @ w - y_s)
    w -= lr * grad
print(f'Gradient descent: intercept={w[0]:.3f}, slope={w[1]:.3f}')

# Method 2: Normal Equation (analytical)
w_ne = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y_s
print(f'Normal Equation:  intercept={w_ne[0]:.3f}, slope={w_ne[1]:.3f}')

# Method 3: sklearn
lr_model = LinearRegression()
lr_model.fit(X_s, y_s)
print(f'sklearn:          intercept={lr_model.intercept_:.3f}, slope={lr_model.coef_[0]:.3f}')
print()
print(f'True values:      intercept=4.000, slope=3.000')
print('All three methods agree — same objective, different solver strategies.')

### Exercise 5.1 — Add regularization
Use `Ridge(alpha=1.0)` from sklearn instead of `LinearRegression`.
Compare the coefficients. When does Ridge help?

In [ ]:
from sklearn.linear_model import Ridge

# TODO: train a Ridge model, compare its R² to LinearRegression on the housing data
# Is Ridge better, worse, or similar here? Why?

---
## Week 10 — Classification & Evaluation (HOML Ch. 3)

In [ ]:
data = load_breast_cancer()
X_c, y_c = data.data, data.target
X_tr, X_te, y_tr, y_te = train_test_split(X_c, y_c, test_size=0.2,
                                           random_state=42, stratify=y_c)
sc = StandardScaler()
X_tr = sc.fit_transform(X_tr); X_te = sc.transform(X_te)

clf = LogisticRegression(max_iter=10000, random_state=42)
clf.fit(X_tr, y_tr)
y_pred_c = clf.predict(X_te)

print(f'Accuracy: {accuracy_score(y_te, y_pred_c):.4f}')
print()
print('Confusion Matrix:')
cm = confusion_matrix(y_te, y_pred_c)
print(cm)
print(f'TN={cm[0,0]}, FP={cm[0,1]}, FN={cm[1,0]}, TP={cm[1,1]}')
print()
print(classification_report(y_te, y_pred_c, target_names=data.target_names))

In [ ]:
# Why accuracy alone is misleading on imbalanced data
# Suppose: 95% of emails are NOT spam (class 0)
# A model that always predicts 'not spam' gets 95% accuracy — but catches 0% of spam!

y_dummy = np.zeros(100)  # always predict class 0
y_true_imbalanced = np.array([1]*5 + [0]*95)  # 5% spam

print(f'Dumb model accuracy: {accuracy_score(y_true_imbalanced, y_dummy):.2f}')
print('But it misses ALL 5 spam emails (recall for class 1 = 0.0).')
print('This is why recall, precision, and F1 exist.')

---
## Self-Check
1. In the cancer classifier, a False Negative means predicting 'benign' when the tumor is malignant.
   Is FN or FP more dangerous here? Which metric should you optimize?
2. RMSE is in the same units as the target. What is the unit of RMSE for the housing model?
3. Why do we use StandardScaler before logistic regression but it doesn't matter as much for trees?

*Your answers here...*